In [1]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

BASE_DIR = Path("../data/01_raw/ADNI")

In [3]:
FILES_TO_CHECK = {
    "diagnosis": "DXSUM.csv",
    "brain_vol": "ADSP_BIOMARKER_SCORE.csv",
    "cognition": "ADSP_COGN_SCORE.csv",
    "genetics": "APOERES.csv",
    "plasma": "PLASMA.csv"
}
PATIENT_IMAGE_DIR = BASE_DIR / "002_S_0295"

In [ ]:
def load_and_inspect_csv(file_path: Path, name: str) -> pd.DataFrame:

    print(f"File: {name.upper()}")
    
    try:
        df = pd.read_csv(file_path)
        
        # Shape analysis
        rows, cols = df.shape
        print(f"Shape: {rows} rows, {cols} columns")
        
        # Duplicate check
        duplicates = df.duplicated().sum()
        print(f"Duplicate Rows: {duplicates}")
        
        # Missing value analysis
        missing_counts = df.isnull().sum()
        
        # Filter for columns that actually have missing values
        missing_data = missing_counts[missing_counts > 0]
        
        if not missing_data.empty:
            # Calculate percentage
            missing_percentage = (missing_data / rows) * 100
            
            # Create a summary dataframe
            missing_summary = pd.DataFrame({
                'Missing Count': missing_data,
                'Percentage': missing_percentage
            })
            
            # Sort by percentage descending to see worst offenders first
            missing_summary = missing_summary.sort_values(by='Percentage', ascending=False)
            
            print(f"Columns with Missing Values ({len(missing_summary)} columns):")
            display(missing_summary.head(15).style.format({'Percentage': '{:.2f}%'}))
        else:
            print("No missing values found.")

        # Quick look
        print("First 3 rows:")
        display(df.head(3))
        
        return df

    except FileNotFoundError:
        print(f"Can't find file at {file_path}")
        return None

In [8]:
def inspect_dicom_structure(directory: Path):
    """
    Walks through a directory to count DICOM files and visualize structure.
    """
    print(f"File structure of: {directory.name}")
    
    if not directory.exists():
        print(f"Directory not found: {directory}")
        return

    dcm_count = 0
    
    # Walk through directory tree
    for root, dirs, files in os.walk(directory):
        level = root.replace(str(directory), '').count(os.sep)
        indent = ' ' * 4 * (level)
        folder_name = os.path.basename(root)
        
        # Count .dcm files in this folder
        current_dcms = [f for f in files if f.endswith('.dcm')]
        count = len(current_dcms)
        dcm_count += count
        
        # Only print folders if they are relevant
        if level < 3: 
            print(f"{indent}|-- {folder_name}/ ({count} .dcm files)")

    print(f"\nTotal DICOM files found: {dcm_count}")

In [ ]:
# Dictionary to store loaded dataframes for easy access later
dfs = {}

# Inspect CSV data
for friendly_name, filename in FILES_TO_CHECK.items():
    full_path = BASE_DIR / filename
    dfs[friendly_name] = load_and_inspect_csv(full_path, friendly_name)

# Inspect image data structure
inspect_dicom_structure(PATIENT_IMAGE_DIR)

File: DIAGNOSIS
Shape: 16032 rows, 41 columns
Duplicate Rows: 0

Columns with Missing Values (33 columns):


,Missing Count,Percentage
DXDDUE,14168,88.37%
DD_CRF_VERSION_LABEL,13601,84.84%
LANGUAGE_CODE,13601,84.84%
HAS_QC_ERROR,13601,84.84%
DXNODEP,12164,75.87%
DXAD,12164,75.87%
DXNORM,12164,75.87%
DXCONFID,12164,75.87%
DXOTHDEM,12164,75.87%
DXPATYP,12164,75.87%



First 3 Rows:


,PHASE,PTID,RID,VISCODE,VISCODE2,EXAMDATE,DIAGNOSIS,DXNORM,DXNODEP,DXMCI,DXMDES,DXMPTR1,DXMPTR2,DXMPTR3,DXMPTR4,DXMPTR5,DXMPTR6,DXMDUE,DXMOTHET,DXDSEV,DXDDUE,DXAD,DXAPP,DXAPROB,DXAPOSS,DXPARK,DXPDES,DXPCOG,DXPATYP,DXDEP,DXOTHDEM,DXODES,DXCONFID,ID,SITEID,USERDATE,USERDATE2,DD_CRF_VERSION_LABEL,LANGUAGE_CODE,HAS_QC_ERROR,update_stamp
0,ADNI1,011_S_0002,2,bl,bl,2005-09-29,1.0,1.0,-4.0,-4.0,-4,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4,-4.0,NaN,-4.0,-4.0,-4,-4,-4.0,-4.0,-4.0,-4.0,NaN,-4.0,-4.0,4.0,2,107,2005-10-01,NaN,NaN,NaN,NaN,2005-10-01 00:00:00
1,ADNI1,011_S_0003,3,bl,bl,2005-09-30,3.0,-4.0,-4.0,-4.0,-4,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4,2.0,NaN,1.0,1.0,1,-4,-4.0,-4.0,-4.0,-4.0,NaN,-4.0,-4.0,3.0,4,107,2005-10-01,NaN,NaN,NaN,NaN,2005-10-01 00:00:00
2,ADNI1,011_S_0005,5,bl,bl,2005-09-30,1.0,1.0,-4.0,-4.0,-4,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4,-4.0,NaN,-4.0,-4.0,-4,-4,-4.0,-4.0,-4.0,-4.0,NaN,-4.0,-4.0,4.0,6,107,2005-10-01,NaN,NaN,NaN,NaN,2005-10-01 00:00:00


File: BRAIN_VOL
Shape: 10260 rows, 212 columns
Duplicate Rows: 0

Columns with Missing Values (4 columns):


...


First 3 Rows:


,PHASE,PTID,RID,VISCODE,GENOTYPE,APTESTDT,APVOLUME,APRECEIVE,APAMBTEMP,APRESAMP,APUSABLE,ID,SITEID,USERDATE,USERDATE2,update_stamp
0,ADNI1,011_S_0002,2,sc,3/3,2005-08-22,5.0,1.0,1.0,0.0,1.0,4.0,107.0,2005-08-23,NaN,2005-08-23 00:00:00
1,ADNI1,011_S_0003,3,sc,3/4,2005-08-22,10.0,1.0,1.0,0.0,1.0,6.0,107.0,2005-08-23,NaN,2005-08-23 00:00:00
2,ADNI1,022_S_0004,4,sc,3/3,2005-08-22,9.2,1.0,1.0,0.0,1.0,8.0,10.0,2005-08-23,NaN,2005-08-23 00:00:00


File: PLASMA
Shape: 2178 rows, 16 columns
Duplicate Rows: 0

No missing values found.

First 3 Rows:


,PHASE,PTID,RID,VISCODE,VISCODE2,EXAMDATE,Primary,Additive,pT217_F,AB42_F,AB40_F,AB42_AB40_F,pT217_AB42_F,NfL_Q,GFAP_Q,update_stamp
0,ADNI1,011_S_0002,2,bl,bl,2005-09-08,BLD,EDT,0.199,27.92,324.12,0.086100,0.007130,14.9,94.6,2024-12-18 09:58:07
1,ADNI1,011_S_0008,8,bl,bl,2005-09-19,BLD,EDT,0.152,27.90,296.53,0.094088,0.005448,26.6,141.1,2024-12-18 09:58:07
2,ADNI2,011_S_0008,8,v41,m120,2015-10-29,BLD,EDT,0.235,7.24,70.00,0.103429,0.032459,55.8,289.1,2024-12-18 09:58:07


File structure of: 002_S_0295
|-- 002_S_0295/ (0 .dcm files)
    |-- MP-RAGE/ (0 .dcm files)
        |-- 2008-07-23_14_51_41.0/ (0 .dcm files)
        |-- 2007-05-25_07_12_36.0/ (0 .dcm files)
        |-- 2006-04-18_08_20_30.0/ (0 .dcm files)
        |-- 2009-05-22_07_09_05.0/ (0 .dcm files)
        |-- 2006-11-02_08_16_44.0/ (0 .dcm files)
        |-- 2009-05-22_07_00_57.0/ (0 .dcm files)
    |-- MP-RAGE_REPEAT/ (0 .dcm files)
        |-- 2006-04-18_08_28_40.0/ (0 .dcm files)
        |-- 2006-11-02_08_25_54.0/ (0 .dcm files)
        |-- 2007-05-25_07_20_37.0/ (0 .dcm files)
        |-- 2008-07-23_14_59_55.0/ (0 .dcm files)

Total DICOM files found: 1660
